## Install PySpark and create a session

In [3]:
!pip install pyspark

In [27]:
from pyspark.sql import SparkSession
from pyspark.sql import DataFrameReader

#create a spark session
spark=SparkSession.builder.appName("AirQuality").getOrCreate()
spark

## Read in data to pyspark dataframe

In [8]:
#Mount Drive
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [29]:
#define folder path
path='/content/drive/MyDrive/AirQualityModel/SiteData'

#import datetime
from datetime import datetime,date,time

#Read a single CSV to check inferred schema
df=spark.read.csv('/content/drive/MyDrive/AirQualityModel/SiteData/AQ_Site1.csv',header=True,inferSchema=True)

In [30]:
#Check the top 5 rows to see that data has read in as expected
df.show(5)

+----------+--------+------------------------------------------+----------------+-----------------------+-------+-------------------+-------+--------------------+---------+---------+----------------+---------------+-------+
|      Date|    Time|PM2.5 particulate matter (Hourly measured)|         Status3|Modelled Wind Direction|Status5|Modelled Wind Speed|Status7|           Site Name| Latitude|Longitude|       Site Type|Local Authority|   Zone|
+----------+--------+------------------------------------------+----------------+-----------------------+-------+-------------------+-------+--------------------+---------+---------+----------------+---------------+-------+
|01/01/2025|01:00:00|                                     7.453|V ugm-3 (Ref.eq)|                  231.4|  N deg|               10.9| N ms-1|Borehamwood Meado...|51.661229| -0.27055|Urban Background|      Hertsmere|Eastern|
|01/01/2025|02:00:00|                                     4.151|V ugm-3 (Ref.eq)|                  231.9

In [38]:
#Check the schema
df.printSchema()

root
 |-- Date: string (nullable = true)
 |-- Time: string (nullable = true)
 |-- PM2.5 particulate matter (Hourly measured): string (nullable = true)
 |-- Status3: string (nullable = true)
 |-- Modelled Wind Direction: string (nullable = true)
 |-- Status5: string (nullable = true)
 |-- Modelled Wind Speed: string (nullable = true)
 |-- Status7: string (nullable = true)
 |-- Site Name: string (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Site Type: string (nullable = true)
 |-- Local Authority: string (nullable = true)
 |-- Zone: string (nullable = true)



#The inferred schema has mis-defined some fields as strings, when they should be dates, times floats and doubles

Define a new schema that better matches the expected data

In [41]:
from pyspark.sql.types import StructType, StructField, StringType, FloatType,DoubleType,TimestampType,DateType

#define the schema
schema = StructType([
    StructField("Date", DateType(), nullable=True),
    StructField("Time", TimestampType(), nullable=True),
    StructField("PM2.5 reading", DoubleType(), nullable=True),
    StructField("PM2.5 Status", StringType(),nullable=True),
    StructField("Wind Direction",FloatType(), nullable=True),
    StructField("Wind Direction Status",StringType(),nullable=True),
    StructField("Wind Speed",FloatType(),nullable=True),
    StructField("wind Speed Status",StringType(),nullable=True),
    StructField("Site Name",StringType(),nullable=True),
    StructField("Latitude",DoubleType(),nullable=True),
    StructField("Longitude",DoubleType(),nullable=True),
    StructField("SiteType",StringType(),nullable=True),
    StructField("Local Authority",StringType(),nullable=True),
    StructField("Zone",StringType(),nullable=True)
    ])

#Read a single CSV to confirm that new schema has been applied
df2=spark.read.csv('/content/drive/MyDrive/AirQualityModel/SiteData/AQ_Site1.csv',header=True,schema=schema)
df2.printSchema()



root
 |-- Date: date (nullable = true)
 |-- Time: timestamp (nullable = true)
 |-- PM2.5 reading: double (nullable = true)
 |-- PM2.5 Status: string (nullable = true)
 |-- Wind Direction: float (nullable = true)
 |-- Wind Direction Status: string (nullable = true)
 |-- Wind Speed: float (nullable = true)
 |-- wind Speed Status: string (nullable = true)
 |-- Site Name: string (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- SiteType: string (nullable = true)
 |-- Local Authority: string (nullable = true)
 |-- Zone: string (nullable = true)



# Having confirmed the new schema is being applied on a single CSV, we now read all the CSVs together into a single dataframe

In [44]:
#read all csv files in folder to single dataframe
df=(spark.read
    .format('csv')
    .option('header',True)
    .schema(schema)
    .load(path))
#Check the schema
df.printSchema()


root
 |-- Date: date (nullable = true)
 |-- Time: timestamp (nullable = true)
 |-- PM2.5 reading: double (nullable = true)
 |-- PM2.5 Status: string (nullable = true)
 |-- Wind Direction: float (nullable = true)
 |-- Wind Direction Status: string (nullable = true)
 |-- Wind Speed: float (nullable = true)
 |-- wind Speed Status: string (nullable = true)
 |-- Site Name: string (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- SiteType: string (nullable = true)
 |-- Local Authority: string (nullable = true)
 |-- Zone: string (nullable = true)

